# 03 — Preprocessing
Clean, encode, scale and split the data.

In [ ]:
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib, os

df = pd.read_csv('../data/processed/listings_eda.csv')
print(f'Loaded: {df.shape}')

In [ ]:
# ── Feature Engineering ────────────────────────────────────────────────────

# 1. amenities_count
#    amenities column is a JSON list string: '["Wifi", "Kitchen", ...]'
#    More amenities = higher quality = higher price potential
def count_amenities(x):
    try:
        return len(ast.literal_eval(x))
    except Exception:
        return 0

df['amenities_count'] = df['amenities'].apply(count_amenities)
print(f"amenities_count  — mean: {df['amenities_count'].mean():.1f}, "
      f"max: {df['amenities_count'].max()}, "
      f"missing: {df['amenities_count'].isna().sum()}")

# 2. host_since_years
#    Years since host joined Airbnb — experienced hosts price more strategically
reference_date = pd.Timestamp('2025-09-01')
df['host_since'] = pd.to_datetime(df['host_since'], errors='coerce')
df['host_since_years'] = (reference_date - df['host_since']).dt.days / 365.25
print(f"host_since_years — mean: {df['host_since_years'].mean():.1f}, "
      f"missing: {df['host_since_years'].isna().sum()}")

# 3. number_of_reviews_ltm — already in dataset, no engineering needed
print(f"number_of_reviews_ltm — missing: {df['number_of_reviews_ltm'].isna().sum()}")

In [ ]:
# ── Feature Selection ──────────────────────────────────────────────────────
# 79 original features → 22 selected + target
#
# PROPERTY SIZE & CAPACITY
#   accommodates      — max guests; strong price driver (SHAP #2)
#   bedrooms          — strongest single price driver (SHAP #1)
#   beds              — complements bedrooms; ~24% missing → median impute
#   bathrooms         — size proxy; ~24% missing → median impute
#   amenities_count   — [ENGINEERED] count of amenities; more = higher quality
#
# LOCATION
#   neighbourhood_cleansed — official district; no NaN (SHAP #5)
#   latitude          — north/south location within Zurich (SHAP #3)
#   longitude         — east/west; together with lat = full geo signal
#
# LISTING TYPE
#   room_type         — entire home vs private/shared room (SHAP #4)
#   property_type     — apartment vs house vs boat etc.
#
# HOST QUALITY
#   host_is_superhost        — Airbnb quality badge; t/f → 1/0
#   host_acceptance_rate     — how often host accepts; '81%' → 0.81
#   host_response_rate       — reliability signal; '100%' → 1.0
#   host_since_years         — [ENGINEERED] years on Airbnb
#   calculated_host_listings_count — professional vs private host
#
# BOOKING & AVAILABILITY
#   minimum_nights    — short-stay vs monthly rental
#   instant_bookable  — convenience; t/f → 1/0
#   availability_365  — days per year listed
#
# REVIEWS & TRUST
#   review_scores_rating  — overall satisfaction; ~24% missing → median impute
#   number_of_reviews     — listing maturity proxy
#   number_of_reviews_ltm — last 12 months; more current than total count
#   reviews_per_month     — activity rate; ~24% missing → median impute
#
# TEMPORAL
#   snapshot_month    — 2025-06 / 2025-09; captures seasonal differences
#
# EXCLUDED
#   estimated_revenue_l365d / estimated_occupancy_l365d — DATA LEAKAGE
#   review_scores_* (6 sub-scores) — ~24% missing, covered by overall rating
#   name / description / host_about — free text, needs NLP
#   listing_url / picture_url / IDs — identifiers only
#   neighbourhood / host_neighbourhood — too many NaN
#   license / calendar_updated — completely empty
# ──────────────────────────────────────────────────────────────────────────

FEATURES = [
    'accommodates', 'bedrooms', 'beds', 'bathrooms', 'amenities_count',
    'neighbourhood_cleansed', 'latitude', 'longitude',
    'room_type', 'property_type',
    'host_is_superhost', 'host_acceptance_rate', 'host_response_rate',
    'host_since_years', 'calculated_host_listings_count',
    'minimum_nights', 'instant_bookable', 'availability_365',
    'review_scores_rating', 'number_of_reviews', 'number_of_reviews_ltm', 'reviews_per_month',
    'snapshot_month',
]
TARGET = 'price'

df = df[FEATURES + [TARGET]].copy()
df = df.dropna(subset=[TARGET])
df = df[df[TARGET] > 0]
print(f'After price filter: {df.shape}')

In [ ]:
# ── Outlier Capping ────────────────────────────────────────────────────────
# Cap top 1% of prices to stabilise training.
# Without capping: a handful of listings at 9000+ CHF destroy RMSE.
# This matches the original project design decision.

cap = df[TARGET].quantile(0.99)
n_capped = (df[TARGET] > cap).sum()
df[TARGET] = df[TARGET].clip(upper=cap)

print(f'Price cap (99th pct): {cap:.0f} CHF')
print(f'Capped {n_capped} listings ({n_capped/len(df)*100:.1f}%)')
print(f'Price range: {df[TARGET].min():.0f} – {df[TARGET].max():.0f} CHF')
print(f'Median: {df[TARGET].median():.0f} CHF  |  Mean: {df[TARGET].mean():.0f} CHF')

In [ ]:
# ── Type Conversions ───────────────────────────────────────────────────────

# Percentage strings → float  ('81%' → 0.81)
for col in ['host_acceptance_rate', 'host_response_rate']:
    df[col] = (df[col].astype(str)
                       .str.replace('%', '', regex=False)
                       .replace('nan', np.nan)
                       .astype(float)) / 100

# Boolean strings → int  (t/f → 1/0)
for col in ['host_is_superhost', 'instant_bookable']:
    df[col] = df[col].map({'t': 1, 'f': 0})

# ── Imputation ─────────────────────────────────────────────────────────────

median_cols = [
    'bedrooms', 'bathrooms', 'beds',
    'reviews_per_month', 'review_scores_rating',
    'host_acceptance_rate', 'host_response_rate',
    'host_is_superhost', 'instant_bookable',
    'host_since_years', 'number_of_reviews_ltm',
]
for col in median_cols:
    n = df[col].isna().sum()
    df[col] = df[col].fillna(df[col].median())
    if n > 0:
        print(f'  Imputed {n:>4} missing in {col}')

# ── Encoding ───────────────────────────────────────────────────────────────

cat_cols = ['neighbourhood_cleansed', 'room_type', 'property_type', 'snapshot_month']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Convert bool → int (pandas 2.0+ compatibility for CSV round-trip)
bool_cols = df.select_dtypes(include='bool').columns.tolist()
if bool_cols:
    df[bool_cols] = df[bool_cols].astype(int)

# Safety check
remaining = df.select_dtypes(include='object').columns.tolist()
if remaining:
    print(f'WARNING: still object dtype: {remaining}')
    df = pd.get_dummies(df, columns=remaining, drop_first=True)
else:
    print('All columns numeric — OK')

print(f'Shape after encoding: {df.shape}')

In [ ]:
# ── Train / Test Split ─────────────────────────────────────────────────────

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train = X_train.copy()
X_test  = X_test.copy()
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# ── Scaling & Save ─────────────────────────────────────────────────────────

scaler = StandardScaler()

# All continuous numeric columns (booleans and one-hot columns excluded)
num_cols = [
    'accommodates', 'bedrooms', 'beds', 'bathrooms', 'amenities_count',
    'latitude', 'longitude',
    'minimum_nights', 'availability_365',
    'number_of_reviews', 'number_of_reviews_ltm', 'reviews_per_month',
    'review_scores_rating',
    'host_acceptance_rate', 'host_response_rate',
    'host_since_years', 'calculated_host_listings_count',
]

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)

print('Splits saved.')
print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train max: {y_train.max():.0f} CHF  |  y_test max: {y_test.max():.0f} CHF')